# B2 — Frozen CLIP

Standalone workflow for this model. Select **Python (ArtFAE .venv)**. Run All defaults to command previews; enable only the actions you intend to execute. GPU work requires the booking in `configs/shoal_booking.yaml` and a successful GPU smoke test. All model selection uses validation data; the held-out test set is excluded.

Run setup, review settings, enable `smoke` and run the smoke cell, then enable `train` (or `evaluate` for B2). New outputs use unique session directories. Interrupt a training cell for graceful checkpointing.

## 1. Kernel and project setup

In VS Code choose **Select Kernel → Python Environments → `/workspace/.venv/bin/python`**.
If that environment needs notebook packages, run in the terminal:

```bash
/workspace/.venv/bin/python -m pip install 'ipykernel>=6,<8' 'nbformat>=5,<6' 'nbclient>=0.10,<1'
```

Use the existing environment. Change `ROOT_OVERRIDE` only if the project moved.


In [ ]:
from pathlib import Path
import copy, hashlib, json, os, shlex, signal, subprocess, sys, time, uuid
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import torch
import yaml
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

ROOT_OVERRIDE = None  # e.g. Path('/workspace/ArtFAE-Net')

def find_project():
    if ROOT_OVERRIDE is not None:
        candidates = [Path(ROOT_OVERRIDE).expanduser().resolve()]
    else:
        here = Path.cwd().resolve()
        candidates = [here, *here.parents]
    for candidate in candidates:
        if (candidate / 'src/artfae').is_dir() and (candidate / 'configs/m1.yaml').is_file():
            return candidate
    raise RuntimeError('Open the notebook inside ArtFAE-Net, or set ROOT_OVERRIDE.')

ROOT = find_project()
PYTHON = Path('/workspace/.venv/bin/python')
if not PYTHON.is_file():
    raise RuntimeError('Set PYTHON to the existing research virtual environment.')
if Path(sys.prefix).resolve() != PYTHON.parent.parent.resolve():
    raise RuntimeError(f'Choose the {PYTHON} kernel. Current kernel prefix: {sys.prefix}')
sys.path.insert(0, str(ROOT / 'src'))
from artfae import EMOTIONS

print('Project:', ROOT)
print('Kernel:', sys.executable)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Class order:', list(EMOTIONS))


## 2. Run settings

The recorded booking is loaded from `configs/shoal_booking.yaml`; change the dates only for an actual approved reservation.
GPU commands refuse to run outside that interval. CPU commands are also available with `DEVICE = 'cpu'`.

For your first GPU run, enable **`smoke` and `m1`**, and execute those sections in order.
Keep cache creation disabled while using the already completed feature cache.
Each notebook session and training run gets a new directory; the original data and CLIP head are preserved.


In [ ]:
DATA_DIR = ROOT / 'data'
CACHE_DIR = ROOT / 'results/features_clip_v1_20261022'
RESULTS_ROOT = ROOT / 'results'
DEVICE = 'cuda'                  # 'cuda' or 'cpu'
SEED = 42
EPOCHS = 20
BATCH_SIZE = 32
ACCUMULATION = 1                 # effective training batch = BATCH_SIZE * ACCUMULATION
PATIENCE = 5

booking = yaml.safe_load((ROOT / 'configs/shoal_booking.yaml').read_text())
BOOKING_START = booking['booking_start']
BOOKING_END = booking['booking_end']

MODEL = 'B2'
RUN = {'tests': False, 'smoke': False, 'build_cache': False, 'compare_export': False, 'evaluate': False}

SESSION = RESULTS_ROOT / 'notebooks' / (datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:6])
for directory in (SESSION, SESSION / 'configs', SESSION / 'jobs'):
    directory.mkdir(parents=True, exist_ok=False)
GPU_SMOKE_REPORT = SESSION / 'gpu_smoke/gpu_smoke.json'
# You may point GPU_SMOKE_REPORT to a previously completed, relevant GPU smoke report.
ACTIVE_PROCESS = None
REGISTRY = []

print('Outputs:', SESSION)
print('Booking:', BOOKING_START, 'to', BOOKING_END)
print('Effective batch:', BATCH_SIZE * ACCUMULATION)
print('Enabled:', [name for name, enabled in RUN.items() if enabled] or 'preview only')


## 3. Command runner

This runner streams output and saves command logs plus exit status. Interrupt a cell to send SIGINT to the trainer;
it preserves its last complete epoch. If saving takes longer than 30 seconds, the process remains attached to
`ACTIVE_PROCESS`; inspect it before launching another job. Resuming replays the incomplete epoch.


In [ ]:
def booking_flags():
    return ['--booking-start', str(BOOKING_START), '--booking-end', str(BOOKING_END)]

def device_flags():
    return ['--device', DEVICE, *(booking_flags() if DEVICE == 'cuda' else [])]

def require_booking():
    start, end = datetime.fromisoformat(str(BOOKING_START)), datetime.fromisoformat(str(BOOKING_END))
    now = datetime.now(timezone.utc)
    if start.tzinfo is None or end.tzinfo is None or not start <= now < end:
        raise RuntimeError(f'Outside the authorized GPU booking: {BOOKING_START} to {BOOKING_END}. Now: {now.isoformat()}')
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA is unavailable. Resolve GPU access or explicitly select DEVICE="cpu".')

def require_gpu_smoke():
    if not GPU_SMOKE_REPORT.is_file():
        raise RuntimeError('Run the GPU smoke section first, or set GPU_SMOKE_REPORT to a passing report.')
    report = json.loads(GPU_SMOKE_REPORT.read_text())
    if not report.get('finite_loss') or not report.get('checkpoint_roundtrip') or report.get('optimizer_steps', 0) < 1 or report.get('cuda_peak_bytes', 0) <= 0:
        raise RuntimeError('The GPU smoke report does not prove a successful CUDA optimizer/checkpoint check.')

def unique_output(label):
    return SESSION / (label + '_' + uuid.uuid4().hex[:8])

def launch(arguments, *, enabled, label, gpu_smoke=False):
    global ACTIVE_PROCESS
    command = [str(PYTHON), *map(str, arguments)]
    print(shlex.join(command))
    if not enabled:
        print('PREVIEW — set the corresponding RUN switch to True to execute.')
        print()
        return None
    if ACTIVE_PROCESS is not None and ACTIVE_PROCESS.poll() is None:
        raise RuntimeError(f'A notebook job is still running: PID {ACTIVE_PROCESS.pid}. Do not start a duplicate.')
    cuda = gpu_smoke or ('--device' in command and command[command.index('--device') + 1] == 'cuda')
    if cuda:
        require_booking()
        if not gpu_smoke:
            require_gpu_smoke()
    token = label + '_' + uuid.uuid4().hex[:8]
    log_path, status_path = SESSION / 'jobs' / f'{token}.log', SESSION / 'jobs' / f'{token}.json'
    status = {'command': command, 'cwd': str(ROOT), 'started_utc': datetime.now(timezone.utc).isoformat(), 'log': str(log_path)}
    env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
    process = subprocess.Popen(command, cwd=ROOT, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    ACTIVE_PROCESS = process
    status.update(pid=process.pid, status='running')
    status_path.write_text(json.dumps(status, indent=2))
    try:
        with log_path.open('w') as log:
            for line in process.stdout:
                log.write(line); log.flush(); print(line, end='')
        returncode = process.wait()
        status.update(status='finished' if returncode == 0 else 'failed', returncode=returncode)
    except KeyboardInterrupt:
        if process.poll() is None:
            process.send_signal(signal.SIGINT)
        try:
            process.wait(timeout=30)
            status.update(status='interrupted', returncode=process.returncode)
        except subprocess.TimeoutExpired:
            status.update(status='interrupt_requested', returncode=None)
            print(f'PID {process.pid} is still saving. ACTIVE_PROCESS.poll() checks its actual state.')
        raise
    finally:
        status['observed_utc'] = datetime.now(timezone.utc).isoformat()
        status_path.write_text(json.dumps(status, indent=2))
        if process.poll() is not None:
            ACTIVE_PROCESS = None
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)
    print()
    print('Saved log:', log_path)
    return status

def script(name, *arguments, enabled, label, gpu_smoke=False):
    return launch([ROOT / 'scripts' / name, *arguments], enabled=enabled, label=label, gpu_smoke=gpu_smoke)

print('Runner ready. All commands use:', PYTHON)


## 4. Inspect the dataset audit and completed cache

The original `results/audit_20261022/audit.json` predates the explicit `something else` → `something_else` mapping.
Use the corrected/decode audit below when reviewing current integrity results. This cell reads metadata only.


In [ ]:
audit_path = ROOT / 'results/audit_decode_20261022/audit.json'
if audit_path.is_file():
    audit = json.loads(audit_path.read_text())
    display(pd.DataFrame(audit['splits']).T)
    print('Artwork-disjoint:', audit.get('artwork_disjoint'))
    print('Reported issues:', len(audit.get('issues', [])))
else:
    print('Run the audit command in TRAINING_GUIDE.md before starting experiments.')

if (CACHE_DIR / 'metadata.json').is_file():
    cache_metadata = json.loads((CACHE_DIR / 'metadata.json').read_text())
    if cache_metadata['classes'] != list(EMOTIONS):
        raise ValueError('Feature cache class order mismatch.')
    print('Cache:', CACHE_DIR)
    for split in ('train', 'val'):
        print(split, 'artworks:', len(cache_metadata['splits'][split]['ids']), 'shards:', len(cache_metadata['splits'][split]['chunks']))
else:
    print('No complete cache found. Select a NEW CACHE_DIR and enable build_cache below.')


## 5. Tests and CPU/GPU smoke

Run these before your first long experiment. The CUDA smoke checks real training images, CLIP feature extraction,
AGF/ECSA forward/backward, optimizer steps, and checkpoint writing. It can run only inside the booking.


In [ ]:
launch(['-m', 'unittest', 'discover', '-s', 'tests', '-v'], enabled=RUN['tests'], label='tests')
if DEVICE == 'cuda':
    smoke_output = unique_output('gpu_smoke') if RUN['smoke'] else GPU_SMOKE_REPORT.parent
    if RUN['smoke']:
        GPU_SMOKE_REPORT = smoke_output / 'gpu_smoke.json'
    script('gpu_smoke.py', '--data-dir', DATA_DIR, '--output', smoke_output, *booking_flags(), enabled=RUN['smoke'], label='gpu_smoke', gpu_smoke=True)
else:
    script('smoke_models.py', '--data-dir', DATA_DIR, '--output', unique_output('cpu_smoke'), enabled=RUN['smoke'], label='cpu_smoke')


## Evaluate the preserved trained head
B2 is already trained. This notebook reuses its original checkpoint and writes a fresh validation report. It does not overwrite or unnecessarily retrain the original model. Use B3–B5 or M1 notebooks for new fusion training experiments.

In [ ]:
CHECKPOINT = ROOT / 'results/clip_b32_full/best_head.pt'
reference_run = unique_output('b2_validation')
script('evaluate_existing_clip.py', '--data-dir', DATA_DIR, '--checkpoint', CHECKPOINT,
       '--output', reference_run, enabled=RUN['evaluate'], label='b2_validation')

## Validation reports
The table below is filtered to this model. Optional comparison export includes all research models for context. Re-run after training finishes.

In [ ]:
def validation_table():
    records = []
    for path in sorted(RESULTS_ROOT.rglob('validation_metrics.json')):
        if any('smoke' in part for part in path.relative_to(RESULTS_ROOT).parts):
            continue
        try:
            metrics = json.loads(path.read_text())
            config_path = path.parent / 'config.json'
            config = json.loads(config_path.read_text()) if config_path.exists() else {}
        except (OSError, json.JSONDecodeError):
            continue  # a concurrent writer may not have finished its report yet
        if 'smoke' in config.get('scope', ''):
            continue
        variant = config.get('variant', 'B1' if 'freeze_backbone' in config else 'B2' if path.parent.name.startswith(('clip_validation', 'b2_validation')) else 'unknown')
        recipe = {key: value for key, value in config.items() if key not in ('seed', 'notebook_label', 'notebook_trial')}
        recipe_id = hashlib.sha256(json.dumps(recipe, sort_keys=True).encode()).hexdigest()[:12]
        records.append({'run': str(path.parent.relative_to(RESULTS_ROOT)), 'path': str(path.parent), 'variant': variant,
                        'recipe': recipe_id, 'seed': config.get('seed', 42), 'macro_f1': metrics.get('macro_f1'),
                        'accuracy': metrics.get('accuracy'), 'soft_ce': metrics.get('soft_ce'),
                        'excluded_ties': metrics.get('excluded_ties'), 'lr': config.get('lr'),
                        'hidden': config.get('architecture', {}).get('hidden'), 'epochs': config.get('epochs')})
    return pd.DataFrame(records, columns=['run','path','variant','recipe','seed','macro_f1','accuracy','soft_ce','excluded_ties','lr','hidden','epochs'])

reports = validation_table()
reports = reports[reports['variant'] == MODEL].sort_values('macro_f1', ascending=False, na_position='last')
display(reports.drop(columns=['path']))
if len(reports):
    # Select the first lexicographic run for each recipe/seed, never the best repeated seed run.
    independent = reports.sort_values('run').drop_duplicates(['variant', 'recipe', 'seed'])
    summary = independent.groupby(['variant', 'recipe']).agg(seeds=('seed','nunique'), mean_macro_f1=('macro_f1','mean'), sample_std=('macro_f1','std')).reset_index()
    display(summary)
    print('Summary uses the first lexicographic report per recipe/seed; duplicate same-seed runs remain in the full table.')

comparison_output = unique_output('comparison')
script('compare_experiments.py', '--results', RESULTS_ROOT, '--data-dir', DATA_DIR,
       '--output', comparison_output, '--plots', '--bootstrap', 1000,
       enabled=RUN['compare_export'], label='comparison')


## 13. Inspect learning curves and per-class results

Set `SELECTED_RUN` to a completed run directory to inspect it. If left as `None`, this cell uses the best
reported M1 validation score in this notebook session only. This is development selection, not a locked final-test protocol.


In [ ]:
SELECTED_RUN = None  # e.g. ROOT / 'results/notebooks/YOUR_SESSION/m1_reference_seed42_RUNID'
current = validation_table()
current = current[(current['variant'] == MODEL) & current['path'].map(lambda p: Path(p).is_relative_to(SESSION))]
if SELECTED_RUN is None and len(current):
    SELECTED_RUN = Path(current.sort_values('macro_f1', ascending=False).iloc[0]['path'])
if SELECTED_RUN is not None:
    SELECTED_RUN = Path(SELECTED_RUN).expanduser().resolve()
    metrics = json.loads((SELECTED_RUN / 'validation_metrics.json').read_text())
    display(pd.DataFrame({'emotion': metrics['classes'], 'precision': metrics['precision'], 'recall': metrics['recall'], 'f1': metrics['f1'], 'support': metrics['support']}))
    history_path = SELECTED_RUN / 'history.json'
    if history_path.exists():
        history = pd.DataFrame(json.loads(history_path.read_text()))
        if len(history):
            fig, axes = plt.subplots(1, 2, figsize=(11, 4))
            axes[0].plot(history['epoch'], history['train_soft_ce'], label='train')
            axes[0].plot(history['epoch'], history['soft_ce'], label='validation')
            axes[0].set(xlabel='Epoch', ylabel='Soft cross-entropy'); axes[0].legend()
            axes[1].plot(history['epoch'], history['macro_f1'])
            axes[1].set(xlabel='Epoch', ylabel='Validation Macro-F1')
            plt.tight_layout(); plt.show()
    if (SELECTED_RUN / 'confusion_matrix.png').exists():
        display(Image(filename=str(SELECTED_RUN / 'confusion_matrix.png')))
else:
    print('No completed run for this model selected yet. Train a model or assign SELECTED_RUN.')


## Record results
Keep the selected recipe, seed, validation Macro-F1, per-class metrics, output directory and checkpoint together. Higher scores are experimental outcomes, not guaranteed improvements. Do not use held-out test results for tuning. M2 faithfulness-aware training is not implemented and has no training notebook.